# 05 — TCOCNN v3 optimieren und mit Plain/V1 vergleichen

Die Hyperparameteroptimierung ist hier ausdrücklich Teil des Versuchs. Entscheidend ist anschließend der faire Vergleich des besten v3 mit dem Plain/V1-TCOCNN: Fehler, Parameterzahl und eingesparte Parameter.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p/'Networks'/'TCOCNNv3.py').exists())
for folder in ['Networks','Evaluation Seminar/Day_02','Evaluation Seminar/Day_03','Evaluation Seminar/Day_04']:
    sys.path.insert(0,str(ROOT/folder))
import numpy as np
import matplotlib.pyplot as plt
from day3_utils import plot_comparison, regression_metrics, show_results

import torch
from skopt import Optimizer
from skopt.space import Categorical, Real
from TCOCNNv3 import TCOCNNv3Class
from day3_utils import prepare_model_data, train_experiment
splits,scaler=prepare_model_data('acetone')
print('Rohdatenquelle:',ROOT/'Data'/'fullData.mat')
print('Input roh:',splits['train']['X'].shape,'; Output acetone:',splits['train']['y'].shape)
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(splits['train']['X'][0,0,:,0])
ax.set(title='Echter Rohzyklus vor jeder Skalierung – SensorA, Kanal 0',xlabel='Zeitindex',ylabel='gespeicherter Rohwert')
ax.grid(True,alpha=.3); plt.show()
EPOCHS=100; SEARCH_TRIALS=24; SEED=42
base=dict(n_filter=32,section_depth=3,kernel=9,stride=4,num_neurons=128,
          drop_out=.15,initial_learning_rate=5e-4,batch_size=64,
          convs_per_block=2,channel_growth=16,residual=True)
rows=[]; predictions={}; histories={}; best_v3=None; best_score=np.inf
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## Architecture knobs

`section_depth` counts convolution/pooling blocks; `convs_per_block` controls depth within a block.
`n_filter` is the initial width and `channel_growth` adds channels in later blocks. `stride` is the
pooling factor in v3, while its convolutions use stride one. `residual` enables shortcuts.
The dense width follows global pooling, so parameter count is not directly comparable to the flattened
head of the plain architecture. We report it alongside RMSE and runtime.

In [ ]:
def run_trial(label,params,v3=True):
    global best_v3,best_score
    model,metadata=train_experiment(splits,params,epochs=EPOCHS,seed=SEED,
                                    model_class=TCOCNNv3Class if v3 else None)
    prediction=model.predict(splits['val']['X_z']).ravel()*scaler['y_std']+scaler['y_mean']
    metrics=regression_metrics(splits['val']['y'],prediction)
    row={'trial':label,**metadata,**metrics,**params}; rows.append(row)
    predictions[label]=prediction; histories[label]=model.history.history
    if v3 and metrics['RMSE_ppb']<best_score:
        best_score=metrics['RMSE_ppb']; best_v3=model
    print(label,'validation RMSE:',round(metrics['RMSE_ppb'],3),flush=True)
    return metrics['RMSE_ppb']
plain={k:v for k,v in base.items() if k not in ['convs_per_block','channel_growth','residual']}
plain['num_neurons']=256
run_trial('Plain reference',plain,v3=False)
run_trial('v3 without residual',{**base,'residual':False})
run_trial('v3 with residual',base)

In [ ]:
dimensions=[Categorical([24,32,48,64],name='n_filter'),Categorical([2,3,4],name='section_depth'),
            Categorical([1,2,3],name='convs_per_block'),Categorical([0,16,32],name='channel_growth'),Categorical([5,9,15],name='kernel'),
            Categorical([2,4],name='stride'),Categorical([128,256],name='num_neurons'),
            Real(0.,.3,name='drop_out'),Real(1e-4,3e-3,prior='log-uniform',name='initial_learning_rate')]
optimizer=Optimizer(dimensions,n_initial_points=6,random_state=SEED)
for trial in range(1,SEARCH_TRIALS+1):
    values=optimizer.ask()
    sample={dimension.name:value for dimension,value in zip(dimensions,values)}
    score=run_trial(f'v3 search {trial}',{**base,**sample})
    optimizer.tell(values,score)
show_results([{k:row[k] for k in ['trial','RMSE_ppb','R2','best_epoch','parameters','seconds']} for row in rows])
show_results([{'trial':row['trial'], 'filters':row['n_filter'], 'blocks':row['section_depth'],
               'convs_per_block':row.get('convs_per_block','-'), 'channel_growth':row.get('channel_growth','-'),
               'residual':row.get('residual','-'), 'dropout':row['drop_out'],
               'kernel':row['kernel'],'pool':row['stride'],'dense':row['num_neurons'],
               'learning_rate':f"{row['initial_learning_rate']:.2e}"} for row in rows])

## Validation comparisons

Compare the controlled residual pair before interpreting the search winner. Common scatter limits
show calibration bias. The parameter/RMSE scatter shows that more parameters need not improve accuracy.
Twenty-four Bayesian trials are an instructional budget, not evidence that one architecture is universally best.

In [ ]:
best_label=min((row for row in rows if row['trial']!='Plain reference'),key=lambda row:row['RMSE_ppb'])['trial']
labels=list(dict.fromkeys(['Plain reference','v3 without residual','v3 with residual',best_label]))
plot_comparison(splits['val']['y'],{label:predictions[label] for label in labels},'Validation: plain, residual ablation, optimized v3')
fig,axes=plt.subplots(1,2,figsize=(14,5))
for label in labels:
    history=histories[label]
    axes[0].plot(np.arange(1,EPOCHS+1),history['val_loss'],label=label)
axes[0].set(xlabel='Epoch',ylabel='Validation standardized MSE',yscale='log'); axes[0].legend()
for row in rows:
    axes[1].scatter(row['parameters'],row['RMSE_ppb'])
    axes[1].annotate(row['trial'],(row['parameters'],row['RMSE_ppb']),fontsize=8)
axes[1].set(xscale='log',xlabel='Trainable parameters',ylabel='Validation RMSE [ppb]')
for ax in axes: ax.grid(True,which='both',alpha=.3)
plt.tight_layout(); plt.show()
plain_row=next(row for row in rows if row['trial']=='Plain reference')
best_row=min((row for row in rows if row['trial']!='Plain reference'),key=lambda row:row['RMSE_ppb'])
saved=plain_row['parameters']-best_row['parameters']
show_results([{'Vergleich':'Plain/V1 TCOCNN','Parameter':plain_row['parameters'],'RMSE_ppb':plain_row['RMSE_ppb'],
               'eingesparte_Parameter':0,'Ersparnis_Prozent':0.0},
              {'Vergleich':'bestes TCOCNN v3','Parameter':best_row['parameters'],'RMSE_ppb':best_row['RMSE_ppb'],
               'eingesparte_Parameter':saved,'Ersparnis_Prozent':100*saved/plain_row['parameters']}])
print(f"Bestes v3 spart {saved:,} Parameter = {100*saved/plain_row['parameters']:.1f}% gegenüber Plain/V1.")


## Freeze the v3 winner and evaluate once

The regular test and extrapolation test have not entered the search. Report both even if extrapolation
is poor. GPU operations may vary slightly between runs despite fixed initialization seeds.

In [ ]:
final=[]
for name in ['test','test_extra']:
    truth=splits[name]['y']
    pred=best_v3.predict(splits[name]['X_z']).ravel()*scaler['y_std']+scaler['y_mean']
    plot_comparison(truth,{best_label:pred,'Train-mean baseline':np.full_like(truth,scaler['y_mean'])},name)
    final.append({'split':name,**regression_metrics(truth,pred)})
show_results(final)
